In [1]:
import pandas

df = pandas.read_csv("./Classwork6Files/Lecture10_files/311_cases.csv")

In [2]:
df.shape

(47540, 48)

In [3]:
df.dtypes

CaseID                                                    int64
Opened                                                   object
Closed                                                   object
Updated                                                  object
Status                                                   object
Status Notes                                             object
Responsible Agency                                       object
Category                                                 object
Request Type                                             object
Request Details                                          object
Address                                                  object
Street                                                   object
Supervisor District                                     float64
Neighborhood                                             object
Police District                                          object
Latitude                                

In [4]:
drop_cols = []
for c in df.columns:
    if "DELETE" in c:
        drop_cols.append(c)

drop_cols

['DELETE - Supervisor Districts',
 'DELETE - Fire Prevention Districts',
 'DELETE - Current Police Districts',
 'DELETE - Zip Codes',
 'DELETE - Police Districts',
 'DELETE - Neighborhoods',
 'DELETE - Neighborhoods_from_fyvs_ahh9',
 'DELETE - 2017 Fix It Zones',
 'DELETE - SF Find Neighborhoods',
 'DELETE - Current Supervisor Districts',
 'DELETE - HSOC Zones']

In [5]:
df = df.drop(columns=drop_cols)

In [6]:
df['Latitude'].describe()

count    47540.000000
mean        30.472348
std         14.910111
min          0.000000
25%         37.727067
50%         37.763965
75%         37.781513
max         37.826729
Name: Latitude, dtype: float64

In [7]:
df['Longitude'].describe()

count    47540.000000
mean       -98.781542
std         48.333740
min       -122.514306
25%       -122.437065
50%       -122.418003
75%       -122.398359
max          0.000000
Name: Longitude, dtype: float64

In [8]:
df = df[df['Latitude'] > 0]
df = df[df['Longitude'] < 0]

In [9]:
df['Latitude'].describe()

count    38357.000000
mean        37.767694
std          0.022601
min         37.708160
25%         37.753700
50%         37.771891
75%         37.784780
max         37.826729
Name: Latitude, dtype: float64

In [10]:
df['Longitude'].describe()

count    38357.000000
mean      -122.430703
std          0.027786
min       -122.514306
25%       -122.445566
50%       -122.422224
75%       -122.411764
max       -122.363781
Name: Longitude, dtype: float64

In [11]:
df.shape

(38357, 37)

In [12]:
df['Opened']

0        11/30/2023 10:59:00 PM
1        11/30/2023 10:56:00 PM
3        11/30/2023 10:41:25 PM
5        11/30/2023 10:36:57 PM
6        11/30/2023 10:35:00 PM
                  ...          
47532    10/31/2023 11:45:00 PM
47533    11/01/2023 06:01:00 PM
47535    11/01/2023 10:01:00 AM
47538    11/01/2023 09:01:00 AM
47539    11/01/2023 09:00:00 AM
Name: Opened, Length: 38357, dtype: object

In [13]:
pandas.to_datetime(df['Opened'])

C:\Users\arjav\AppData\Local\Temp\ipykernel_61272\1181775674.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  pandas.to_datetime(df['Opened'])


0       2023-11-30 22:59:00
1       2023-11-30 22:56:00
3       2023-11-30 22:41:25
5       2023-11-30 22:36:57
6       2023-11-30 22:35:00
                ...        
47532   2023-10-31 23:45:00
47533   2023-11-01 18:01:00
47535   2023-11-01 10:01:00
47538   2023-11-01 09:01:00
47539   2023-11-01 09:00:00
Name: Opened, Length: 38357, dtype: datetime64[ns]

In [14]:
df['Opened'] = pandas.to_datetime(df['Opened'])
df['Closed'] = pandas.to_datetime(df['Closed'])

C:\Users\arjav\AppData\Local\Temp\ipykernel_61272\2786706544.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['Opened'] = pandas.to_datetime(df['Opened'])


In [15]:
df['OpenTime'] = df['Closed'] - df['Opened']

In [16]:
df['OpenTime']

0        0 days 07:53:43
1       14 days 09:12:00
3        0 days 07:19:52
5        0 days 07:24:13
6        0 days 09:44:00
              ...       
47532    0 days 05:31:13
47533    0 days 21:45:00
47535    1 days 02:49:00
47538    5 days 01:18:00
47539    1 days 01:46:00
Name: OpenTime, Length: 38357, dtype: timedelta64[ns]

In [17]:
df_neighborhood = df.groupby("Neighborhood").agg(
    {
        "OpenTime": "mean",
        "CaseID": "count"
    }
)

In [18]:
df_neighborhood.to_csv("./311_neighborhood.csv")

In [19]:
import arcgis

In [20]:
df.columns

Index(['CaseID', 'Opened', 'Closed', 'Updated', 'Status', 'Status Notes',
       'Responsible Agency', 'Category', 'Request Type', 'Request Details',
       'Address', 'Street', 'Supervisor District', 'Neighborhood',
       'Police District', 'Latitude', 'Longitude', 'Point', 'Source',
       'Media URL', 'BOS_2012', 'SF Find Neighborhoods',
       'Current Police Districts', 'Current Supervisor Districts',
       'Analysis Neighborhoods',
       'Civic Center Harm Reduction Project Boundary',
       'Fix It Zones as of 2017-11-06 ', 'Invest In Neighborhoods (IIN) Areas',
       'Fix It Zones as of 2018-02-07',
       'CBD, BID and GBD Boundaries as of 2017',
       'Central Market/Tenderloin Boundary', 'Areas of Vulnerability, 2016',
       'Central Market/Tenderloin Boundary Polygon - Updated',
       'HSOC Zones as of 2018-06-05', 'OWED Public Spaces',
       'Parks Alliance CPSI (27+TL sites)', 'Neighborhoods', 'OpenTime'],
      dtype='object')

In [21]:
df_311 = pandas.DataFrame.spatial.from_xy(
    df = df,
    x_column = 'Longitude',
    y_column = 'Latitude',
    sr = 4326
)

In [22]:
df_cbg = pandas.DataFrame.spatial.from_featureclass(
    "./Classwork6Files/Lecture10_files/Tutorial_08.gdb/Census_Block_Groups"
)

In [23]:
df_join = df_311.spatial.join(
    right_df = df_cbg,
    how = 'left',
    op = 'intersects'
)

In [24]:
df_join.columns

Index(['CaseID', 'Opened', 'Closed', 'Updated', 'Status', 'Status Notes',
       'Responsible Agency', 'Category', 'Request Type', 'Request Details',
       'Address', 'Street', 'Supervisor District', 'Neighborhood',
       'Police District', 'Latitude', 'Longitude', 'Point', 'Source',
       'Media URL', 'BOS_2012', 'SF Find Neighborhoods',
       'Current Police Districts', 'Current Supervisor Districts',
       'Analysis Neighborhoods',
       'Civic Center Harm Reduction Project Boundary',
       'Fix It Zones as of 2017-11-06 ', 'Invest In Neighborhoods (IIN) Areas',
       'Fix It Zones as of 2018-02-07',
       'CBD, BID and GBD Boundaries as of 2017',
       'Central Market/Tenderloin Boundary', 'Areas of Vulnerability, 2016',
       'Central Market/Tenderloin Boundary Polygon - Updated',
       'HSOC Zones as of 2018-06-05', 'OWED Public Spaces',
       'Parks Alliance CPSI (27+TL sites)', 'Neighborhoods', 'OpenTime',
       'SHAPE', 'index_right', 'OBJECTID', 'geoid'],
      

In [25]:
df_cbg_summary = df_join.groupby("geoid").agg(
    {
        "CaseID": "count",
        "OpenTime": "mean"
    }
)

In [26]:
df_cbg_summary['OpenTime'] = df_cbg_summary['OpenTime'].dt.days

In [27]:
df_cbg_summary = df_cbg.merge(
    df_cbg_summary,
    how = 'left',
    left_on = 'geoid',
    right_on = 'geoid'
)

In [28]:
gis = arcgis.GIS()
cbg_map = gis.map("San Francisco, CA")

df_to_map = df_cbg_summary[pandas.notna(df_cbg_summary.OpenTime)]

df_to_map.spatial.plot(map_widget=cbg_map)

cbg_map.legend.enabled = True

c:\Users\arjav\AppData\Local\ESRI\conda\envs\gis222venv\Lib\site-packages\arcgis\features\geo\_accessor.py:1606: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  self._data[col] = array


In [29]:
renderer_manager = cbg_map.content.renderer(0)

smart_mapper = renderer_manager.smart_mapping()

smart_mapper.class_breaks_renderer(
    break_type = 'color',
    field = 'OpenTime',
    classification_method = 'natural-breaks',
    num_classes = 5,
)

In [30]:
cbg_map

Map(center=[4548404.498098655, -13627750.041010315], extent={'xmin': -13660589.290794332, 'ymin': 4506937.6933…

In [31]:
import pandas
import requests
import json
import datetime

In [32]:
now = datetime.datetime.now()
now

datetime.datetime(2025, 11, 3, 14, 22, 47, 621829)

In [33]:
start_date = now - datetime.timedelta(days=7)
start_date

datetime.datetime(2025, 10, 27, 14, 22, 47, 621829)

In [34]:
midnight = datetime.time()

start_day_midnight = datetime.datetime.combine(start_date, midnight)

start_day_midnight

datetime.datetime(2025, 10, 27, 0, 0)

In [35]:
field_list = [
    "service_request_id",
    "requested_datetime",
    "status_notes",
    "service_name",
    "service_subtype",
    "lat",
    "long",
    "neighborhoods_sffind_boundaries",
    "source",
    "supervisor_district",
    "media_url",
    "point"
]

fields = "$select=" + ",".join(field_list)
fields

'$select=service_request_id,requested_datetime,status_notes,service_name,service_subtype,lat,long,neighborhoods_sffind_boundaries,source,supervisor_district,media_url,point'

In [36]:
url = "https://data.sfgov.org/resource/vw6y-z8j6.json"

In [39]:
formatted_start_day = start_day_midnight.strftime("%Y-%m-%dT%H:%M:%S.%f")
formatted_start_day

'2025-10-27T00:00:00.000000'

In [40]:
where = f"$where=status_description='Open' and requested_datetime>'{formatted_start_day}'"
where

"$where=status_description='Open' and requested_datetime>'2025-10-27T00:00:00.000000'"

In [42]:
full_url = url+"?"+fields+"&"+where
full_url

"https://data.sfgov.org/resource/vw6y-z8j6.json?$select=service_request_id,requested_datetime,status_notes,service_name,service_subtype,lat,long,neighborhoods_sffind_boundaries,source,supervisor_district,media_url,point&$where=status_description='Open' and requested_datetime>'2025-10-27T00:00:00.000000'"

In [43]:
response = requests.get(full_url)
response

<Response [200]>

In [44]:
response_text = response.text

In [46]:
print(type(response_text))
print(len(response_text))
response_text[0:1000]

<class 'str'>
543210


'[{"service_request_id":"101002865304","requested_datetime":"2025-10-27T01:08:55.000","status_notes":"in progress","service_name":"Street and Sidewalk Cleaning","service_subtype":"garbage_and_debris","lat":"37.76445049","long":"-122.4212554","neighborhoods_sffind_boundaries":"Mission","source":"Web","supervisor_district":"9.0","media_url":{"url":"https://sanfrancisco.form.us.empro.verintcloudservices.com/form/auto/download_attachments?caseid=101002865304&formref=K2502658"},"point":{"latitude":"37.76445049","longitude":"-122.4212554","human_address":"{\\"address\\": \\"\\", \\"city\\": \\"\\", \\"state\\": \\"\\", \\"zip\\": \\"\\"}"}}\n,{"service_request_id":"101002865688","requested_datetime":"2025-10-27T07:18:53.000","status_notes":"accepted","service_name":"Parking Enforcement","service_subtype":"other_illegal_parking","lat":"37.75522834","long":"-122.40886646","neighborhoods_sffind_boundaries":"Mission","source":"Phone","supervisor_district":"9.0","point":{"latitude":"37.75522834",

In [47]:
results_list = json.loads(response_text)
results_list[0:2]

[{'service_request_id': '101002865304',
  'requested_datetime': '2025-10-27T01:08:55.000',
  'status_notes': 'in progress',
  'service_name': 'Street and Sidewalk Cleaning',
  'service_subtype': 'garbage_and_debris',
  'lat': '37.76445049',
  'long': '-122.4212554',
  'neighborhoods_sffind_boundaries': 'Mission',
  'source': 'Web',
  'supervisor_district': '9.0',
  'media_url': {'url': 'https://sanfrancisco.form.us.empro.verintcloudservices.com/form/auto/download_attachments?caseid=101002865304&formref=K2502658'},
  'point': {'latitude': '37.76445049',
   'longitude': '-122.4212554',
   'human_address': '{"address": "", "city": "", "state": "", "zip": ""}'}},
 {'service_request_id': '101002865688',
  'requested_datetime': '2025-10-27T07:18:53.000',
  'status_notes': 'accepted',
  'service_name': 'Parking Enforcement',
  'service_subtype': 'other_illegal_parking',
  'lat': '37.75522834',
  'long': '-122.40886646',
  'neighborhoods_sffind_boundaries': 'Mission',
  'source': 'Phone',
  's

In [48]:
df = pandas.DataFrame(results_list)
df.head()

,service_request_id,requested_datetime,status_notes,service_name,service_subtype,lat,long,neighborhoods_sffind_boundaries,source,supervisor_district,media_url,point
0,101002865304,2025-10-27T01:08:55.000,in progress,Street and Sidewalk Cleaning,garbage_and_debris,37.76445049,-122.4212554,Mission,Web,9.0,{'url': 'https://sanfrancisco.form.us.empro.ve...,"{'latitude': '37.76445049', 'longitude': '-122..."
1,101002865688,2025-10-27T07:18:53.000,accepted,Parking Enforcement,other_illegal_parking,37.75522834,-122.40886646,Mission,Phone,9.0,NaN,"{'latitude': '37.75522834', 'longitude': '-122..."
2,101002865966,2025-10-27T08:03:35.000,open,RPD General,structural_maintenance,37.769059013,-122.4809487,Golden Gate Park,Phone,1.0,NaN,"{'latitude': '37.769059013', 'longitude': '-12..."
3,101002866447,2025-10-27T08:55:08.000,open,Sewer,sewer_odor,37.74153217,-122.46307827,Laguna Honda,Web,7.0,NaN,"{'latitude': '37.74153217', 'longitude': '-122..."
4,101002866562,2025-10-27T09:05:49.000,accepted,Graffiti Public,not_offensive,37.74566906,-122.40291231,Produce Market,Web,10.0,{'url': 'https://sanfrancisco.form.us.empro.ve...,"{'latitude': '37.74566906', 'longitude': '-122..."


In [49]:
def street_cleaning(value):
    if value == "Street and Sidewalk Cleaning":
        return 1
    else:
        return 0
    
print(street_cleaning("Street and Sidewalk Cleaning"))
print(street_cleaning("some other string"))

1
0


In [51]:
df['street_sidewalk_cleaning'] = df['service_name'].apply(street_cleaning)

df['graffiti'] = df['service_name'].apply(street_cleaning)

df['total_cases'] = 1

df[['service_name','street_sidewalk_cleaning','graffiti','total_cases']]

,service_name,street_sidewalk_cleaning,graffiti,total_cases
0,Street and Sidewalk Cleaning,1,1,1
1,Parking Enforcement,0,0,1
2,RPD General,0,0,1
3,Sewer,0,0,1
4,Graffiti Public,0,0,1
...,...,...,...,...
995,Graffiti Public,0,0,1
996,Graffiti Public,0,0,1
997,Graffiti Public,0,0,1
998,Graffiti Private,0,0,1
